#Scoring the Accident Images 

In this notebook, we'll show how to score the incoming images of damaged vehicles as they flow into the system to come up with a damage severity score using the model that we previously training and promoted to the production.

The end results are stored in the `accident_images` delta table

*Note: we could also have these transformations available in a Spark Declarative Pipelines. Open [02.2-EXTRA-Batch-Scoring-DLT]($./EXTRA-DLT-inference/02.2-EXTRA-Batch-Scoring-DLT) to see how it's done.*


<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F02-Data-Science-ML%2F02.2-Batch-Scoring&demo_name=lakehouse-fsi-smart-claims&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-fsi-smart-claims%2F02-Data-Science-ML%2F02.2-Batch-Scoring&version=1">

In [0]:
%pip install mlflow==2.20.2

  Using cached mlflow-2.20.2-py3-none-any.whl.metadata (30 kB)
  Using cached mlflow_skinny-2.20.2-py3-none-any.whl.metadata (31 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
Using cached mlflow-2.20.2-py3-none-any.whl (28.4 MB)
Using cached mlflow_skinny-2.20.2-py3-none-any.whl (6.0 MB)
Using cached docker-7.1.0-py3-none-any.whl (147 kB)
Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
Using cached graphql_relay-3.2.0-py3-none-any.whl (16 kB)
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 2.21.3
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-f10921f3-bc21-4cee-8135-f6ede1c3e717
    Can't uninstall 'mlflow-skinny'. No files were found to uninstall.
Note: you may need to restart the

In [0]:
%run ../_resources/00-setup

run done
USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_fsi_smart_claims`


data already existing.


In [0]:
from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository
import mlflow
# Use the Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")
# download model requirement from remote registry
requirements_path = ModelsArtifactRepository(f"models:/{catalog}.{db}.dbdemos_claims_damage_level@prod").download_artifacts(artifact_path="requirements.txt") 

In [0]:
%pip install -r $requirements_path
dbutils.library.restartPython()

  Using cached transformers-4.49.0-py3-none-any.whl.metadata (44 kB)
  Using cached torch-2.5.1-cp312-cp312-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.20.1-cp312-cp312-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached accelerate-1.4.0-py3-none-any.whl.metadata (19 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_

In [0]:
%run ../_resources/00-setup $reset_all_data=false

run done
USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_fsi_smart_claims`


data already existing.


## Incrementally ingest the raw incoming images

New images can typically land in a cloud storage (S3/ADLS/GCS), mounted within Unity Catalog using Volumes. 

Let's start by ingesting them and saving them as a Delta Lake table 

In [0]:
volume_path = f"/Volumes/{catalog}/{db}/{volume_name}"

(spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "binaryFile")
            .option("cloudFiles.schemaLocation", f"{volume_path}/checkpoint/images_shema")
            .load(f"{volume_path}/Accidents/images")
            .withColumn("image_name", F.regexp_extract(F.col("path"), r".*/(.*?.jpg)", 1))
      .writeStream
            .option("checkpointLocation", f"{volume_path}/checkpoint/images")
            .trigger(availableNow=True)
            .table("raw_accident_image")).awaitTermination()

display(spark.table("raw_accident_image").limit(10))

path,modificationTime,length,content,image_name
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)",2_High.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/5_Low.jpg,2025-10-23T22:19:55Z,1878130,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLXA4b1BiN3dqTTJKS0Q= (truncated), ...)",5_Low.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/4_High.jpg,2025-10-23T22:19:55Z,1844884,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLWFveHpiR1lQUGprcVQ= (truncated), ...)",4_High.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/3_High.jpg,2025-10-23T22:19:55Z,1748114,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLWl3dDZWbnFLb0dsOUQ= (truncated), ...)",3_High.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/4_Low.jpg,2025-10-23T22:19:55Z,1744522,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLXA4b1BiN3dqTTJKS0Q= (truncated), ...)",4_Low.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/4_Medium.jpg,2025-10-23T22:19:55Z,1672837,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVRZRDk4RFQ4REd3bm8= (truncated), ...)",4_Medium.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/5_High.jpg,2025-10-23T22:19:55Z,1672683,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLWFveHpiR1lQUGprcVQ= (truncated), ...)",5_High.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/5_Medium.jpg,2025-10-23T22:19:55Z,1655724,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLTRMWThCazl0a1Zadzg= (truncated), ...)",5_Medium.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_Medium.jpg,2025-10-23T22:19:55Z,1603316,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLTBxMmJWR0dpNkRvRlo= (truncated), ...)",2_Medium.jpg
dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/3_Medium.jpg,2025-10-23T22:19:55Z,1602524,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVRZRDk4RFQ4REd3bm8= (truncated), ...)",3_Medium.jpg


In [0]:
(spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaLocation", f"{volume_path}/checkpoint/images_m_shema")
            .load(f"{volume_path}/Accidents/metadata")
      .writeStream
            .option("checkpointLocation", f"{volume_path}/checkpoint/images_m")
            .trigger(availableNow=True)
            .table("raw_accident_metadata")).awaitTermination()
            
display(spark.table("raw_accident_metadata").limit(10))

image_name,image_id,claim_no,chassis_no,_rescued_data
4_High.jpg,10,32d5b8fd-ed32-4370-aac3-9e7d8fcb499d,JN8BT05Y87W111874,null
4_High.jpg,10,1ee11edb-6169-4c24-80f8-110461f59777,6G1EK54HX9L237163,null
4_High.jpg,10,27cfb515-af2f-4b91-a4b5-9c57836879af,6G1EK54HX9L237163,null
4_High.jpg,10,05e83630-4488-441a-a566-953384630511,5N1AL0MM7DC333224,null
4_High.jpg,10,4abc9897-e0a4-4705-83b5-4fddd707b0fc,JTHBG262582012524,null
4_High.jpg,10,5eabe7d8-9d70-4406-a815-4ac38234581e,JN1FN61C08W093833,null
4_High.jpg,10,9d011028-c7e1-4aa7-83af-4de41c98ccd8,JNRAR07Y7XW066462,null
4_High.jpg,10,091ff093-4291-44e1-a614-58603b16f91b,5N1AR2M51FC631760,null
4_High.jpg,10,b6d72036-8cb2-4c7b-87c9-cf75163b95b3,JTDBW923794025112...,null
4_High.jpg,10,42053276-1b49-4412-8c51-86f278b38fd2,JTDBW923794025112...,null


## Score the damaged vehicle image to determine damage severity

Our claim images are now added to our tables and easily accessible. Let's load our model from Unity Catalog to score the damage. 

In [0]:
import mlflow
model_name = "dbdemos_claims_damage_level"

mlflow.set_registry_uri('databricks-uc')

#Loading the model from UC
predict_damage_udf = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{catalog}.{db}.{model_name}@prod")
spark.udf.register("predict_damage", predict_damage_udf)
columns = predict_damage_udf.metadata.get_input_schema().input_names()

2025/11/03 22:18:43 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/11/03 22:18:43 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


## Test inferences

In [0]:
%sql 
SELECT image_name, predict_damage(content) as damage_prediction, content FROM raw_accident_image LIMIT 10

image_name,damage_prediction,content
2_High.jpg,"List(major, 0.47762542963027954)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)"
5_Low.jpg,"List(minor, 0.47230780124664307)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLXA4b1BiN3dqTTJKS0Q= (truncated), ...)"
4_High.jpg,"List(minor, 0.4809276759624481)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLWFveHpiR1lQUGprcVQ= (truncated), ...)"
3_High.jpg,"List(major, 0.4861834943294525)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLWl3dDZWbnFLb0dsOUQ= (truncated), ...)"
4_Low.jpg,"List(minor, 0.45382049679756165)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLXA4b1BiN3dqTTJKS0Q= (truncated), ...)"
4_Medium.jpg,"List(major, 0.4883517920970917)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVRZRDk4RFQ4REd3bm8= (truncated), ...)"
5_High.jpg,"List(major, 0.5004648566246033)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLWFveHpiR1lQUGprcVQ= (truncated), ...)"
5_Medium.jpg,"List(minor, 0.46317827701568604)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLTRMWThCazl0a1Zadzg= (truncated), ...)"
2_Medium.jpg,"List(minor, 0.49351760745048523)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLTBxMmJWR0dpNkRvRlo= (truncated), ...)"
3_Medium.jpg,"List(minor, 0.48290374875068665)","List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVRZRDk4RFQ4REd3bm8= (truncated), ...)"


In [0]:
raw_images = (spark.read.table("raw_accident_image")
                   .withColumn("damage_prediction", predict_damage_udf(*columns)))

#Only process 1k claims for the demo to run faster
metadata = spark.table("raw_accident_metadata").orderBy(F.rand()).limit(1000)

raw_images.join(metadata, on="image_name").write.mode('overwrite').saveAsTable("accident_images")

In [0]:
display(spark.table("accident_images").limit(10))

image_name,path,modificationTime,length,content,damage_prediction,image_id,claim_no,chassis_no,_rescued_data
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,0d67469e-127e-4704-a747-f3f8ed1aead7,KMHS281D1KU178638,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,90035736-7748-4088-8a75-8241b2cf3562,KMHJT81E2FU010417,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,f5c3d29f-a0c8-48f3-96ba-4522fdb14c5f,6G1EK52B87L866741,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,6fee8211-bc67-4bd8-b5a6-6322418e2b50,LB3774KZ1LH003062,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,ba28aae5-5f2f-4776-8513-54a01745dd36,MHFBT9F37F6024652,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,bfa9b3df-3c5e-49c2-8929-8d53e9a17206,JTDER23EX63160019,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,7563f805-2cda-48c1-9ccc-68a1b1a7984f,RKLBL9HE8E5206426,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,fde074ef-f4f8-45c4-9b86-eeec06e06c86,NLAFC5637KW001605,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,363d555e-c0bc-4258-9c38-991ddb37328d,MHFXX43G6F0057990,null
2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-


## Real time inference

While this use-case is working with batch inferences (consuming incremental new data in a stream), we could also deploy our model behind a [Serverless model endpoint](#mlflow/endpoints). 

Images can be sent as base64 data over the endpoint. For more details on how to do that, you can run `dbdemos.install('computer-vision-pcb')`.

# Add telematics and accident data to the claims & policy data

Telematics data is joined with Claims and Policy data to monitor the behavior of the driver before the accident. End results are stored into "claim_policy_accident" delta table

In [0]:
%sql
CREATE OR REPLACE TABLE claim_policy_accident AS 
  SELECT
    t.*,
    a.* EXCEPT (chassis_no, claim_no)
  FROM
    claim_policy_telematics t
    JOIN accident_images a USING(claim_no)

num_affected_rows,num_inserted_rows



## Conclusion

In this notebook, we demonstrated how to <b> retrieve the model </b> from Unity Catalog and run inferences to <b> score </b> on new image data and persist the results back into delta tables. 

Telematics data, accident image data, claims & policy data </b> are all joined together to provide a 360 view of the accident scene to the claims investigation officer to <b>reconstruct the scene</b> and make a decision on what to do next. Eg. release funds, authorize the car for repairs, approve rental loaner car or send for further investigation.

Open notebook [02.3-Dynamic-Rule-Engine]($./02.3-Dynamic-Rule-Engine) to see how dynamic rules can be implemented to start processing our claims faster based on this information.